In [1]:
# We will need Qibo to write our quantum circuit
# We will need Qibo to define an observable
from qibo import Circuit, gates, hamiltonians, set_backend
from qibo.symbols import Z

# We will need mpstab to construct a surrogate
from mpstab import HSMPO
from mpstab.engines import QuimbEngine, StimEngine
from mpstab.models.ansatze import HardwareEfficient

In [45]:
n = 1000

circ = Circuit(nqubits=n)
circ.add(gates.H(0))
[circ.add(gates.CNOT(q, q+1)) for q in range(n - 1)]

print(circ.summary())

Circuit depth = 1000
Total number of gates = 1000
Number of qubits = 1000
Most common gates:
cx: 999
h: 1


In [46]:
HSMPO?

Init signature:
HSMPO(
    ansatz: Union[mpstab.models.ansatze.Ansatz, qibo.models.circuit.Circuit],
    max_bond_dimension: int = None,
    initial_state: qibo.models.circuit.Circuit = None,
) -> None
Docstring:     
Construct an hybrid stabilizer MPO surrogate of a given quantum circuit.

The tensor-network part is now engine-pluggable via the TensorNetworkEngine API.
File:           ~/Documents/PostDoc/mpstab/src/mpstab/evolutors/hsmpo.py
Type:           type
Subclasses:     

In [47]:
# Construct an hybrid stabilizers MPO surrogate
surr = HSMPO(circ)
surr.set_engines(
    stab_engine=StimEngine(),
    tn_engine=QuimbEngine(),
)

In [48]:
surr.expectation(
    observable="Z"*n
)

np.float64(1.0)

### Supporting generic observables

We leverage Qibo's Symbolic Hamiltonians. 

In [11]:
# Defining a pretty random Hamiltonian
nqubits = 12
ham_form = 0.
for i in range(nqubits - 1):
    ham_form += (-1) ** i * (i + 0.3) * Z(i) * Z(i+1) 
h = hamiltonians.SymbolicHamiltonian(ham_form)

ham_form

0.3*Z0*Z1 - 1.3*Z1*Z2 + 10.3*Z10*Z11 + 2.3*Z2*Z3 - 3.3*Z3*Z4 + 4.3*Z4*Z5 - 5.3*Z5*Z6 + 6.3*Z6*Z7 - 7.3*Z7*Z8 + 8.3*Z8*Z9 - 9.3*Z9*Z10

In [12]:
ans = HardwareEfficient(nqubits=nqubits, nlayers=3)
hs = HSMPO(ansatz=ans, max_bond_dimension=32)

In [13]:
expval = hs.expectation(observable=h)
expval

/Users/robbiati/Documents/tac/lib/python3.12/site-packages/autoray/autoray.py:96: RuntimeWarning: divide by zero encountered in matmul
  return func(*args, **kwargs)
/Users/robbiati/Documents/tac/lib/python3.12/site-packages/autoray/autoray.py:96: RuntimeWarning: overflow encountered in matmul
  return func(*args, **kwargs)
/Users/robbiati/Documents/tac/lib/python3.12/site-packages/autoray/autoray.py:96: RuntimeWarning: invalid value encountered in matmul
  return func(*args, **kwargs)


1.364642914320863

In [14]:
set_backend("numpy")

# double check with Qibo
h.expectation_from_state(ans.circuit().state())

[Qibo 0.3.0|INFO|2026-02-25 10:13:56]: Using numpy backend on /CPU:0
/Users/robbiati/Documents/PostDoc/qibo-related/qibo/src/qibo/backends/abstract.py:399: RuntimeWarning: divide by zero encountered in matmul
  return self.engine.matmul(array_1, array_2, **kwargs)
/Users/robbiati/Documents/PostDoc/qibo-related/qibo/src/qibo/backends/abstract.py:399: RuntimeWarning: overflow encountered in matmul
  return self.engine.matmul(array_1, array_2, **kwargs)
/Users/robbiati/Documents/PostDoc/qibo-related/qibo/src/qibo/backends/abstract.py:399: RuntimeWarning: invalid value encountered in matmul
  return self.engine.matmul(array_1, array_2, **kwargs)
/Users/robbiati/Documents/PostDoc/qibo-related/qibo/src/qibo/backends/abstract.py:1581: RuntimeWarning: divide by zero encountered in matmul
  hstate = hamiltonian @ state
/Users/robbiati/Documents/PostDoc/qibo-related/qibo/src/qibo/backends/abstract.py:1581: RuntimeWarning: overflow encountered in matmul
  hstate = hamiltonian @ state
/Users/robbi

np.float64(1.3646429141864134)

#### The game is about the percentage of Stabilizerness

In [40]:
import time
from copy import deepcopy

nqubits = 32

ans = HardwareEfficient(nqubits=nqubits, nlayers=7)
print(ans.circuit.summary())

surr = HSMPO(ansatz=ans, max_bond_dimension=248)

it = time.time()
# Approx 95% of Magic gates are here replaced with Clifford gates
surr.expectation_from_partition(observable="Z"*nqubits, replacement_probability=0.95)
ft = time.time() - it

print(f"Time with high stabilizerness", ft)

it = time.time()
# Approx 5% of Magic gates are here replaced with Clifford gates
surr.expectation_from_partition(observable="Z"*nqubits, replacement_probability=0.05)
ft = time.time() - it

print(f"Time with low stabilizerness", ft)

Circuit depth = 231
Total number of gates = 448
Number of qubits = 32
Most common gates:
ry: 224
cz: 224
Time with high stabilizerness 1.6811890602111816
Time with low stabilizerness 80.78697109222412


#### Mpstab as backend provider

In [28]:
set_backend("mpstab")

[Qibo 0.3.0|INFO|2026-02-25 10:17:02]: Using MPStabBackend() backend on /CPU:0


In [29]:
# Defining a pretty random Hamiltonian
nqubits = 12
ham_form = 0.
for i in range(nqubits - 1):
    ham_form += (-1) ** i * (i + 0.3) * Z(i) * Z(i+1) 
h = hamiltonians.SymbolicHamiltonian(ham_form)

ham_form

0.3*Z0*Z1 - 1.3*Z1*Z2 + 10.3*Z10*Z11 + 2.3*Z2*Z3 - 3.3*Z3*Z4 + 4.3*Z4*Z5 - 5.3*Z5*Z6 + 6.3*Z6*Z7 - 7.3*Z7*Z8 + 8.3*Z8*Z9 - 9.3*Z9*Z10

In [ ]:
# Now mpstab supports SymbolicHamiltonians as input
ham = hamiltonians.SymbolicHamiltonian(nqubits=nqubits, form=ham_form)

In [32]:
# Using a shortcut here
ans = HardwareEfficient(nqubits=nqubits)

In [34]:
# Using Qibo's interface
ham.expectation(ans.circuit)

np.float64(-3.9469831257193135)